# 🔥 Mojo/MAX M0 hardware probe — Colab T4 (issue #57)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vedmalex/higgs-local-test/blob/main/notebooks/mojo_max_m0_t4.ipynb)

Отдельный, лёгкий блокнот только для M0 — не задевает TTS/STT/Qwen стек из
`higgs_colab_benchmark.ipynb`. Запускает **тот же, неизменный**
`docs/research/mojo-max/m0_smoke_test.mojo`, который уже прошёл на M1 (Apple GPU) —
здесь та же программа исполняется на T4 (`sm_75`, Turing).

Результат нужен для честной записи в issue #57 (M0 verdict) и
`docs/research/mojo-max/m0-results.md`: PASSED/BLOCKED по каждому пункту, реальные
числа, а не гипотеза. T4 не имеет аппаратной поддержки BF16 (в отличие от M1) —
это уже задокументированный факт с Apple-прогона, здесь его нужно подтвердить/
опровергнуть на самом железе.

**Быстрая проверка**: весь блокнот — это только установка `pixi`+`modular` и один
запуск `mojo run`. Ничего лишнего не ставится, TTS/Qwen/STT сюда не подключены.


## 1. GPU и драйвер


In [ ]:
!nvidia-smi


In [ ]:
import subprocess

# MAX/Mojo на CUDA требует driver >=580 (см. issue #57). nvidia-smi печатает версию
# драйвера в шапке таблицы выше; здесь просто извлекаем её программно для гейта.
smi = subprocess.run(["nvidia-smi", "--query-gpu=driver_version,name,compute_cap",
                       "--format=csv,noheader"], capture_output=True, text=True)
print(smi.stdout.strip() or smi.stderr)

DRIVER_VERSION = smi.stdout.strip().split(",")[0].strip() if smi.returncode == 0 else None
if DRIVER_VERSION:
    major = int(DRIVER_VERSION.split(".")[0])
    if major < 580:
        print(f"\n⚠️  Драйвер {DRIVER_VERSION} < 580 — MAX может отказаться работать на GPU.\n"
              "Escape hatch из issue #57: переменная MODULAR_NVPTX_COMPILER_PATH.\n"
              "Ниже это фиксируется как известный риск, а не тихо игнорируется.")
    else:
        print(f"\nДрайвер {DRIVER_VERSION} >= 580 — требование MAX выполнено.")


## 2. Установка pixi + Mojo/MAX (стабильный канал 26.5)


In [ ]:
!curl -fsSL https://pixi.sh/install.sh | bash
import os
os.environ["PATH"] = f"{os.path.expanduser('~/.pixi/bin')}:{os.environ['PATH']}"
!pixi --version


In [ ]:
!mkdir -p /content/mojo-probe-t4
!cd /content/mojo-probe-t4 && pixi init . -c https://conda.modular.com/max/ -c conda-forge
!cd /content/mojo-probe-t4 && pixi add modular
!cd /content/mojo-probe-t4 && pixi run mojo --version && pixi run max --version


## 3. Скрипт пробника из репозитория

`m0_smoke_test.mojo` написан так, что GPU- и CPU-секции независимы: провал GPU не
обрывает CPU-числа, и наоборот. Берём файл как есть с ветки/тега, указанного в
`REPO_REF` — по умолчанию `main` (после мерджа PR #63), без локальных правок.


In [ ]:
REPO_URL = "https://github.com/vedmalex/higgs-local-test.git"
REPO_REF = "main"  # поменять на research/57-m0-gpu-unblocked, если PR #63 ещё не смержен

!rm -rf /content/higgs-local-test
!git clone --quiet --depth 1 --branch "$REPO_REF" "$REPO_URL" /content/higgs-local-test
!ls /content/higgs-local-test/docs/research/mojo-max/


## 4. Запуск пробника на T4


In [ ]:
import subprocess, time
from pathlib import Path

script = "/content/higgs-local-test/docs/research/mojo-max/m0_smoke_test.mojo"
out_path = Path("/content/m0-output-t4.txt")

start = time.time()
proc = subprocess.run(
    ["pixi", "run", "mojo", "run", script],
    cwd="/content/mojo-probe-t4", capture_output=True, text=True,
)
elapsed = time.time() - start

output = proc.stdout + proc.stderr
out_path.write_text(output, encoding="utf-8")
print(output)
print(f"\n--- exit code {proc.returncode}, {elapsed:.1f}s ---")
print(f"Сохранено: {out_path}")


## 5. Итог

Скачайте `/content/m0-output-t4.txt` (панель Files слева, или `files.download`
ниже) и положите в `docs/research/mojo-max/m0-output-t4.txt` локально — затем
обновите `m0-results.md` и оставьте честный комментарий в issue #57 с реальными
числами (PASSED/BLOCKED по каждому пункту GPU-секции), как это уже сделано для M1.

Ничего не приукрашивать: если GPU-секция падает на T4 (например, из-за требования
к драйверу или отсутствия поддержки `sm_75` в текущем MAX), это такой же валидный,
документируемый результат M0, как и BLOCKED-вердикт на macOS 14.6.1 до обновления.


In [ ]:
from google.colab import files
files.download("/content/m0-output-t4.txt")
